# Runtime

## Tổng quan

Bên dưới nền tảng, hàm [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) của LangChain hoạt động dựa trên runtime của LangGraph.

LangGraph cung cấp một đối tượng [`Runtime`](https://reference.langchain.com/python/langgraph/runtime/Runtime) chứa các thông tin sau:

1. **Context**: các thông tin tĩnh như user id, kết nối cơ sở dữ liệu, hoặc các dependency (phụ thuộc) khác dùng cho một lần gọi agent.
2. **Store**: một instance của [`BaseStore`](https://reference.langchain.com/python/langchain-core/stores/BaseStore) được sử dụng làm [bộ nhớ dài hạn](https://docs.langchain.com/oss/python/langchain/long-term-memory).
3. **Stream writer**: một đối tượng dùng để stream thông tin thông qua chế độ stream `"custom"`.
4. **Execution info**: thông tin định danh và thử lại (retry) cho lần thực thi hiện tại (thread ID, run ID, attempt number).
5. **Server info**: metadata đặc thù của máy chủ khi chạy trên LangGraph Server (assistant ID, graph ID, người dùng đã xác thực).

<div class="alert alert-success">

Context của runtime cung cấp cơ chế **dependency injection** (tiêm phụ thuộc) cho các tool và middleware của bạn. Thay vì hardcode (gắn cứng) các giá trị hoặc sử dụng global state (trạng thái toàn cục), bạn có thể inject các dependency của runtime (như kết nối database, user ID, hoặc cấu hình) khi gọi agent. Điều này giúp các tool của bạn dễ test (kiểm thử), tái sử dụng và linh hoạt hơn.

</div>

Bạn có thể truy cập thông tin runtime bên trong các [tool](https://docs.langchain.com/oss/python/langchain/runtime#inside-tools) và [middleware](https://docs.langchain.com/oss/python/langchain/runtime#inside-middleware).

## Truy cập

Khi tạo một agent bằng [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent), bạn có thể chỉ định một `context_schema` để định nghĩa cấu trúc của `context` được lưu trong [`Runtime`](https://reference.langchain.com/python/langgraph/runtime/Runtime) của agent.

Khi gọi (invoke) agent, hãy truyền argument `context` kèm theo cấu hình tương ứng cho lần chạy đó:

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class Context:
    user_name: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Tên tôi là gì?"}]},
    context=Context(user_name="John Smith")
)

### Bên trong tool

Bạn có thể truy cập thông tin runtime bên trong các tool để:

* Truy cập context
* Đọc hoặc ghi vào bộ nhớ dài hạn
* Ghi dữ liệu vào [stream tùy chỉnh](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates) (ví dụ: báo cáo tiến trình / cập nhật của tool)

Sử dụng parameter `ToolRuntime` để truy cập đối tượng [`Runtime`](https://reference.langchain.com/python/langgraph/runtime/Runtime) bên trong một tool.

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class Context:
    user_id: str

@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:
    """Lấy tùy chọn email của người dùng từ kho lưu trữ."""
    user_id = runtime.context.user_id

    preferences: str = "Người dùng muốn bạn viết một email ngắn gọn và lịch sự."
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            preferences = memory.value["preferences"]

    return preferences

### Execution info và server info bên trong tool

Truy cập thông tin định danh thực thi (thread ID, run ID) thông qua `runtime.execution_info`, và metadata đặc thù của máy chủ (assistant ID, người dùng đã xác thực) thông qua `runtime.server_info` khi chạy trên LangGraph Server:

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def context_aware_tool(runtime: ToolRuntime) -> str:
    """Một tool sử dụng thông tin máy chủ và thực thi."""
    # Truy cập thread ID và run ID
    info = runtime.execution_info
    print(f"Luồng: {info.thread_id}, Lần chạy: {info.run_id}")

    # Truy cập thông tin máy chủ (chỉ khả dụng trên LangGraph Server)
    server = runtime.server_info
    if server is not None:
        print(f"Trợ lý: {server.assistant_id}")
        if server.user is not None:
            print(f"Người dùng: {server.user.identity}")

    return "hoàn thành"

`server_info` sẽ có giá trị `None` khi không chạy trên LangGraph Server (ví dụ: trong quá trình phát triển ở local).

<div class="alert alert-info">

Yêu cầu `deepagents>=0.5.0` (hoặc `langgraph>=1.1.5`) để sử dụng `runtime.execution_info` và `runtime.server_info`.

</div>

### Bên trong middleware

Bạn có thể truy cập thông tin runtime trong middleware để tạo các prompt động, chỉnh sửa tin nhắn, hoặc điều khiển hành vi của agent dựa trên context của người dùng.

Sử dụng parameter `Runtime` để truy cập đối tượng [`Runtime`](https://reference.langchain.com/python/langgraph/runtime/Runtime) bên trong các [node-style hook](https://docs.langchain.com/oss/python/langchain/middleware/custom#node-style-hooks). Đối với [wrap-style hook](https://docs.langchain.com/oss/python/langchain/middleware/custom#wrap-style-hooks), đối tượng `Runtime` sẽ có sẵn bên trong parameter [`ModelRequest`](https://reference.langchain.com/python/langchain/agents/middleware/types/ModelRequest).

In [ ]:
from dataclasses import dataclass

from langchain.messages import AnyMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model, after_model
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_name: str

# Prompt động
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    system_prompt = f"Bạn là một trợ lý hữu ích. Hãy xưng hô với người dùng là {user_name}."
    return system_prompt

# Hook before_model (trước mô hình)
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Đang xử lý yêu cầu cho người dùng: {runtime.context.user_name}")
    return None

# Hook after_model (sau mô hình)
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Đã hoàn thành yêu cầu cho người dùng: {runtime.context.user_name}")
    return None

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Tên tôi là gì?"}]},
    context=Context(user_name="John Smith")
)

### Execution info và server info bên trong middleware

Các middleware hook cũng có thể truy cập `runtime.execution_info` và `runtime.server_info`:

In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime


@before_model
def auth_gate(state: AgentState, runtime: Runtime) -> dict | None:
    """Chặn những người dùng chưa xác thực khi chạy trên LangGraph Server."""
    server = runtime.server_info
    if server is not None and server.user is None:
        raise ValueError("Yêu cầu xác thực")
    print(f"Luồng: {runtime.execution_info.thread_id}")
    return None

<div class="alert alert-info">

Yêu cầu `deepagents>=0.5.0` (hoặc `langgraph>=1.1.5`).

</div>